# Embedding
A small version of FastText is used (100 dimensions) in order to make the embedding faster. The pipeline is the subsequent:
- An embedding model is trained on each snapshot (as defined in ```\understanding```). 
- Stopwords and words with < 3 characters are removed since they're considered noise, and the remaining words are transformed into vectors.

In [ ]:
from pathlib import Path
import fasttext
from collections import Counter
import numpy as np
import nltk
from nltk.corpus import stopwords
import os
import multiprocessing
from sklearn.metrics.pairwise import cosine_similarity
import random
import matplotlib.pyplot as plt
import sys

1. The models are trained on the different snapshots.

In [ ]:
# Model creation

for file in os.listdir("../lemmas/snaps"):
    i = 1
    input_file = Path(f"../lemmas/snaps/{file}")  
    model_out = Path(f"fasttext_snap{i}_lite")   # prefisso modello output

    # TRAINING FASTTEXT LEGGERO


    print("Avvio training FastText...")

    model = fasttext.train_unsupervised(
        input=str(input_file),
        model="skipgram",
        dim=100,
        minn=3,
        maxn=5,
        wordNgrams=1,
        epoch=5,
        bucket=2000000,
        thread=multiprocessing.cpu_count()
    )

    # salva modello binario pronto per embedding
    model.save_model(f"{model_out}.bin")
    print(f"Modello binario salvato come {model_out}.bin")
    i += 1


Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap1.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap2.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap3.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap4.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap5.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap6.txt_lite.bin
Avvio training FastText...
Modello binario salvato come fasttext_snap1_snap7.txt_lite.bin


2. Stopwords are removed with **nltk**, as well as words shorter than 3 characters (considered noise). Moreover, frequencies of all the words are computed for each snapshot and those with a frequency slower than the 10th-percentile are removed. Finally, word embeddings are generated. The resulting files are

- ```vocab_freq_snap{n}.csv``` = words (no stopwords and characters < 3) associated with their frequency. Less frequent words are present here.
- ```fasttext_snap{n}_filt.vec``` = word embeddings of words with frquency > threshold (10th-percentile).


In [ ]:
from pathlib import Path
from collections import Counter
import re
from nltk.corpus import stopwords

stopwords_it = set(stopwords.words("italian"))
digit_pattern = re.compile(r"\d")

def stream_clean_tokens(file_path: Path):
    """
    Generatore di token puliti:
    - lowercase
    - no stopwords
    - len >= 3
    - no numeri
    - no token con cifre
    """
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            for tok in line.lower().split():
                if len(tok) < 3:
                    continue
                if digit_pattern.search(tok):
                    continue
                if tok in stopwords_it:
                    continue
                yield tok


In [ ]:
import numpy as np

SNAPS_DIR = Path("../lemmas/snaps")
PERCENTILI = [ 75, 80, 85, 90, 95, 99]

for snap in sorted(SNAPS_DIR.iterdir()):
    if not snap.is_file():
        continue

    print(f"\n=== {snap.name} ===")

    freq = Counter(stream_clean_tokens(snap))

    if not freq:
        print("Nessun token valido dopo pulizia.")
        continue

    counts = np.array(list(freq.values()))

    print(f"Token unici (puliti): {len(freq)}")
    print(f"Token totali (puliti): {counts.sum()}")

    for p in PERCENTILI:
        val = np.percentile(counts, p)
        print(f"p{p:>2}: {val:.2f}")



=== snap1.txt ===
Token unici (puliti): 348915
Token totali (puliti): 33412756
p75: 6.00
p80: 9.00
p85: 16.00
p90: 34.00
p95: 114.00
p99: 1256.86

=== snap2.txt ===
Token unici (puliti): 277540
Token totali (puliti): 32247508
p75: 7.00
p80: 11.00
p85: 19.00
p90: 42.00
p95: 138.00
p99: 1619.61

=== snap3.txt ===
Token unici (puliti): 279788
Token totali (puliti): 31613164
p75: 7.00
p80: 11.00
p85: 20.00
p90: 43.00
p95: 140.00
p99: 1584.00

=== snap4.txt ===
Token unici (puliti): 346349
Token totali (puliti): 32281831
p75: 6.00
p80: 10.00
p85: 18.00
p90: 38.00
p95: 125.00
p99: 1266.00

=== snap5.txt ===
Token unici (puliti): 373441
Token totali (puliti): 31856973
p75: 5.00
p80: 9.00
p85: 16.00
p90: 34.00
p95: 113.00
p99: 1155.00

=== snap6.txt ===
Token unici (puliti): 520455
Token totali (puliti): 33600403
p75: 2.00
p80: 3.00
p85: 6.00
p90: 15.00
p95: 60.00
p99: 731.00

=== snap7.txt ===
Token unici (puliti): 389210
Token totali (puliti): 31740120
p75: 3.00
p80: 5.00
p85: 10.00
p90: 25

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import re
from nltk.corpus import stopwords

# --- setup stopwords e pattern numeri ---
stopwords_it = set(stopwords.words("italian"))
digit_pattern = re.compile(r"\d")

def clean_tokens(file_path: Path):
    """
    Generatore di token puliti:
    - lowercase
    - no stopwords
    - lunghezza >= 3
    - niente numeri
    - niente token con cifre
    """
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            for tok in line.lower().split():
                if len(tok) < 3:
                    continue
                if digit_pattern.search(tok):
                    continue
                if tok in stopwords_it:
                    continue
                yield tok

def tokens_above_percentile(file_path: Path, percentile: float = 90):
    """
    Restituisce:
    - numero totale di token sopra il percentile
    - numero di token unici sopra il percentile
    - soglia di frequenza corrispondente
    """
    freq = Counter(clean_tokens(file_path))
    if not freq:
        return 0, 0, 0.0

    counts = np.array(list(freq.values()))
    soglia = np.percentile(counts, percentile)

    # token sopra o uguali alla soglia
    vocab_filtrato = {w: c for w, c in freq.items() if c >= soglia}
    n_unici = len(vocab_filtrato)
    n_totali = sum(vocab_filtrato.values())

    return n_totali, n_unici, soglia

# --- Esempio di utilizzo ---
SNAPS_DIR = Path("../lemmas/snaps")

for snap in sorted(SNAPS_DIR.iterdir()):
    if not snap.is_file():
        continue

    n_totali, n_unici, soglia_freq = tokens_above_percentile(snap, 90)
    print(f"{snap.name}: token totali sopra 90° percentile = {n_totali}, "
          f"token unici sopra 90° percentile = {n_unici}, "
          f"soglia frequenza = {soglia_freq:.2f}")


snap1.txt: token totali sopra 90° percentile = 32286594, token unici sopra 90° percentile = 35402, soglia frequenza = 34.00
snap2.txt: token totali sopra 90° percentile = 31171036, token unici sopra 90° percentile = 28091, soglia frequenza = 42.00
snap3.txt: token totali sopra 90° percentile = 30508420, token unici sopra 90° percentile = 28108, soglia frequenza = 43.00
snap4.txt: token totali sopra 90° percentile = 31059947, token unici sopra 90° percentile = 35074, soglia frequenza = 38.00
snap5.txt: token totali sopra 90° percentile = 30674693, token unici sopra 90° percentile = 37812, soglia frequenza = 34.00
snap6.txt: token totali sopra 90° percentile = 32765461, token unici sopra 90° percentile = 53355, soglia frequenza = 15.00
snap7.txt: token totali sopra 90° percentile = 30851591, token unici sopra 90° percentile = 39086, soglia frequenza = 25.00


In [5]:
import fasttext
import numpy as np
from pathlib import Path
from collections import Counter
import nltk
from nltk.corpus import stopwords
import csv

# --- 1. SETUP GLOBALE ---
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stopwords_it = set(stopwords.words('italian'))

def stream_tokens_clean(file_path):
    """
    Generatore di token puliti dallo snap:
    - rimuove stopwords
    - rimuove parole <3 caratteri
    - rimuove numeri
    """
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            for token in line.lower().split():
                if len(token) < 3:
                    continue
                if any(char.isdigit() for char in token):
                    continue
                if token in stopwords_it:
                    continue
                yield token

# --- 2. CICLO PRINCIPALE PER GLI SNAP ---
for n in range(1, 8):
    print(f"\n--- Processing Snap {n} ---")

    # A. Caricamento modello FastText
    model_path = Path(f"fasttext_snap{n}_lite.bin")
    if not model_path.exists():
        print(f"Modello {model_path} non trovato, salto.")
        continue

    print("Caricamento modello...", end="")
    model = fasttext.load_model(str(model_path))
    vocab_set = set(model.get_words())
    dim = model.get_dimension()
    print(f" OK (vocab={len(vocab_set)})")

    # B. Pulizia parole e calcolo frequenze sullo snap
    input_file = Path(f"../lemmas/snaps/snap{n}.txt")
    print("Calcolo frequenze parole pulite...", end="")
    freq = Counter(stream_tokens_clean(input_file))
    print(f" OK (token unici puliti={len(freq)})")

    if not freq:
        print("Nessun token valido trovato.")
        continue

    # C. Soglia top 10% (90° percentile)
    counts_array = np.array(list(freq.values()))
    soglia_90 = np.percentile(counts_array, 90)

    top_tokens_snap = {w for w, c in freq.items() if c >= soglia_90}
    print(f"Top 10% snap: {len(top_tokens_snap)} parole (soglia freq = {soglia_90:.2f})")

    # D. Filtra vocab FastText con top_tokens_snap
    final_words = vocab_set & top_tokens_snap
    print(f"Parole finali nel modello: {len(final_words)}")

    # E. Esportazione vettori .vec normalizzati
    output_vec = Path(f"fasttext_snap{n}_filt.vec")
    with output_vec.open("w", encoding="utf-8") as f_vec:
        f_vec.write(f"{len(final_words)} {dim}\n")
        for w in final_words:
            vec = model.get_word_vector(w)
            norm = np.linalg.norm(vec)
            if norm == 0:
                continue
            vec = vec / norm
            vec_str = " ".join(f"{x:.6f}" for x in vec)
            f_vec.write(f"{w} {vec_str}\n")

    # F. Export CSV frequenze (top 10% dello snap)
    output_csv = Path(f"frequencies_all/vocab_freq_snap{n}.csv")
    with output_csv.open("w", newline="", encoding="utf-8") as f_csv:
        writer = csv.writer(f_csv)
        writer.writerow(["word", "frequency"])
        for w in sorted(top_tokens_snap, key=lambda x: -freq[x]):
            writer.writerow([w, freq[w]])

    print(f"Export completato: {output_vec.name}, {output_csv.name}")
    print(f"Stats Snap {n}: Vocab FT={len(vocab_set)} → Top10% Snap={len(top_tokens_snap)} → Finale={len(final_words)}")



--- Processing Snap 1 ---
Caricamento modello... OK (vocab=102410)
Calcolo frequenze parole pulite... OK (token unici puliti=348915)
Top 10% snap: 35402 parole (soglia freq = 34.00)
Parole finali nel modello: 35311
Export completato: fasttext_snap1_filt.vec, vocab_freq_snap1.csv
Stats Snap 1: Vocab FT=102410 → Top10% Snap=35402 → Finale=35311

--- Processing Snap 2 ---
Caricamento modello... OK (vocab=89271)
Calcolo frequenze parole pulite... OK (token unici puliti=277540)
Top 10% snap: 28091 parole (soglia freq = 42.00)
Parole finali nel modello: 28019
Export completato: fasttext_snap2_filt.vec, vocab_freq_snap2.csv
Stats Snap 2: Vocab FT=89271 → Top10% Snap=28091 → Finale=28019

--- Processing Snap 3 ---
Caricamento modello... OK (vocab=90228)
Calcolo frequenze parole pulite... OK (token unici puliti=279788)
Top 10% snap: 28108 parole (soglia freq = 43.00)
Parole finali nel modello: 28031
Export completato: fasttext_snap3_filt.vec, vocab_freq_snap3.csv
Stats Snap 3: Vocab FT=90228 →

Example

In [8]:
import fasttext
import multiprocessing
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import random
import matplotlib.pyplot as plt


vec_file = "word_embeddings_cleaned_ok/fasttext_snap1_filt.vec"

# dizionario parola -> vettore
word2vec = {}
with open(vec_file, encoding="utf-8") as f:
    header = f.readline()  # contiene num_words e dim, da ignorare
    for line in f:
        parts = line.rstrip().split()
        word = parts[0]
        vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
        word2vec[word] = vec


parola1 = "economico"
parola2 = "politico"

if parola1 in word2vec and parola2 in word2vec:
    vec1 = word2vec[parola1].reshape(1, -1)
    vec2 = word2vec[parola2].reshape(1, -1)

    sim = cosine_similarity(vec1, vec2)[0][0]
    distanza = 1 - sim

    print(f"Similarità tra '{parola1}' e '{parola2}': {sim:.4f}")
    print(f"Distanza coseno: {distanza:.4f}")
else:
    print(f"Una delle due parole non è presente: {parola1}, {parola2}")


Similarità tra 'economico' e 'politico': 0.7106
Distanza coseno: 0.2894
